# Labo 39 · Carnet Colab — Mettre un modèle en ligne

Un modèle qui reconnaît les chiffres manuscrits est mis en ligne de trois façons : une interface Gradio, une API Flask et un conteneur Docker. Le GPU n'est pas nécessaire.

**Avant de commencer** : *Fichier → Enregistrer une copie dans Drive*.

À la fin, recopie tes résultats dans la case « Carnet Colab » de la page du labo.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio", "flask"], check=False)
import json, time, hashlib, pickle, threading
import numpy as np
import sklearn
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X / 16, y, test_size=0.25, random_state=0)
modele = LogisticRegression(max_iter=2000).fit(X_tr, y_tr)
VERSION = "1.0.0"
octets = pickle.dumps(modele)
with open("modele.pkl", "wb") as f:
    f.write(octets)
print("exactitude :", round(modele.score(X_te, y_te), 3), "| empreinte :", hashlib.sha256(octets).hexdigest()[:12])

## 1. Une interface Gradio

Le dessin arrive sous forme d'image (un tableau de pixels). Il faut le ramener au format des données d'entraînement : **8 × 8 pixels**, valeurs de 0 à 1, chiffre clair sur fond foncé. C'est le **prétraitement**, qui doit être identique en production et à l'entraînement.

**À toi** : complète `vers_8x8(image)`, puis lance l'interface et dessine quelques chiffres. Lesquels sont mal reconnus? Pourquoi?

In [ ]:
import gradio as gr
from PIL import Image

def vers_8x8(image):
    # image : dictionnaire de Gradio (clé "composite") ou tableau; à toi
    pass

def reconnaitre(image):
    x = vers_8x8(image)
    proba = modele.predict_proba(x.reshape(1, -1))[0]
    return {str(c): float(p) for c, p in zip(modele.classes_, proba)}

## 2. Une API Flask

**À toi** : complète la route `/predire`. Elle reçoit `{"pixels": [64 nombres de 0 à 1]}`, valide l'entrée (une liste de 64 nombres entre 0 et 1, aucun autre champ), retourne `400` avec la liste des erreurs, ou `200` avec le chiffre, sa probabilité et la version. Chaque appel est ajouté au `journal` avec sa latence. Le serveur tourne dans un fil d'exécution du carnet; on l'appelle avec `requests`.

In [ ]:
from flask import Flask, request, jsonify
import requests
app = Flask(__name__)
journal = []

@app.post("/predire")
def predire():
    debut = time.perf_counter()
    donnees = request.get_json(silent=True) or {}
    # À toi : valider, prédire, journaliser, répondre
    return jsonify({}), 501

## 3. Le conteneur

**À toi** : écris les deux fichiers d'un conteneur pour cette API : `requirements.txt` (versions **figées**, celles de ce carnet) et `Dockerfile`. Tu ne peux pas lancer Docker dans Colab; sur un ordinateur où Docker est installé, `docker build -t chiffres:1.0.0 .` puis `docker run -p 8000:8000 chiffres:1.0.0` suffiraient.

In [ ]:
# À toi

## 4. Analyser le journal

**À toi** : envoie 300 requêtes à l'API (des chiffres du test, plus quelques requêtes invalides), puis calcule à partir du `journal` la latence p50 et p95 (en ms) et le taux de refus. Le service respecte-t-il l'objectif « 95 % des requêtes en moins de 50 ms »?

In [ ]:
# À toi

## À remettre

Recopie dans la case « Carnet Colab » de la page du labo : la prédiction de l'interface pour deux chiffres que tu as dessinés (bien ou mal reconnus, et pourquoi), les réponses de l'API aux trois requêtes de la partie 2, ton Dockerfile, et la latence p50 et p95 mesurée dans le journal.